<a href="https://colab.research.google.com/github/LauraaRoman/PhishAid/blob/main/phishaid.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q transformers torch pandas gradio

### Data Synchronization: Integrating the External Knowledge Base
This section is critical for the modularity and scalability of the PhishAid project. By fetching the remedies.csv file directly from our GitHub repository, we implement a Data-Driven Architecture.

Why we use this approach:

Decoupled Logic: The AI engine (the code) is separated from the remediation data (the CSV). This allows the Data Architect to update security protocols on GitHub without needing to modify or redeploy the source code.

Real-time Collaboration: My teammate can refine the remediation steps in the CSV file independently. The next time the script runs, it will automatically pull the most recent version using the !wget command.

Dynamic Knowledge Base: This ensures that PhishAid remains relevant as new phishing techniques emerge, simply by expanding the external dataset.

In [6]:
!wget -O remedies.csv https://raw.githubusercontent.com/LauraaRoman/PhishAid/refs/heads/main/remedies.csv
import pandas as pd
try:
    remedies_df = pd.read_csv('remedies.csv')
    print(f"System: Successfully synchronized PhishAid with the GitHub database ({len(remedies_df)} rules loaded).")
except Exception as e:
    print(f"Error: Could not load the CSV file. Please check the raw URL. Details: {e}")

--2026-05-05 14:18:35--  https://raw.githubusercontent.com/LauraaRoman/PhishAid/refs/heads/main/remedies.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.108.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1646 (1.6K) [text/plain]
Saving to: ‘remedies.csv’

remedies.csv        100%[===================>]   1.61K  --.-KB/s    in 0s      

2026-05-05 14:18:35 (24.0 MB/s) - ‘remedies.csv’ saved [1646/1646]

System: Successfully synchronized PhishAid with the GitHub database (14 rules loaded).


### AI Model Integration: Multi-Task Learning
In this stage, we connect our application to the HuggingFace Hub to load two specialized pre-trained models. This approach allows PhishAid to perform multi-dimensional analysis on any suspicious message.

Tasks identified for this project:

Phishing Detection (Text Classification): We use a BERT-based model fine-tuned specifically to distinguish between legitimate communications and phishing attempts.

Romanian Sentiment Analysis: To enhance detection, we integrate a model from ReaderBench that analyzes the tone of the message in Romanian. This helps identify "Social Engineering" tactics, such as creating a sense of urgency or fear to manipulate the user.

This dual-model setup ensures a higher level of accuracy and provides contextual information for the final security report.

In [7]:
from transformers import pipeline
phish_detector = pipeline("text-classification", model="ealvaradob/bert-finetuned-phishing")
sentiment_analyser = pipeline("text-classification", model="readerbench/ro-sentiment")

print("Success: AI Models have been successfully downloaded and loaded into memory.")

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Success: AI Models have been successfully downloaded and loaded into memory.


###Core Logic: Translating AI Predictions into Remediation
This is the "brain" of the application. The engine takes the raw predictions from the AI models and correlates them with our dynamic knowledge base (remedies.csv).

The Decision Workflow:

1. Data Ingestion: The engine accepts a message from the user.

2. AI Processing: It runs the text through both the Phishing and Sentiment models.

3. Contextual Mapping: If a threat is detected, it performs a keyword search within the CSV file to find the specific "First Aid" steps required for that category (e.g., Banking, Social Media).

4. Report Generation: It outputs a human-readable safety report, fulfilling the project's goal of being a "Translator" of cyber threats.

In [8]:
def phishaid_engine(message):
    phish_res = phish_detector(message)[0]
    tone_res = sentiment_analyser(message)[0]

    prediction_label = phish_res['label'].lower()
    confidence_score = phish_res['score']

    tone_mapping = {
        "LABEL_0": "Urgent / Negative (Panic-based)",
        "LABEL_1": "Neutral / Informative",
        "LABEL_2": "Positive / Friendly"
    }
    friendly_tone = tone_mapping.get(tone_res['label'], "Undefined")

    report = f"PHISHAID SECURITY DIAGNOSIS \n"
    report += f"Classification: {prediction_label.upper()} (Confidence: {confidence_score:.2f})\n"
    report += f"Message Tone: {friendly_tone}\n\n"

    if prediction_label in ["phishing", "label_1"]:
        report += "ALERT: THIS MESSAGE IS HIGHLY SUSPICIOUS!\n"
        report += "EMERGENCY ACTION PLAN (from GitHub Database):\n"

        found_actions = []
        for index, row in remedies_df.iterrows():
            if str(row['Keyword']).lower() in message.lower():
                found_actions.append(f"- [{row['Category']}]: {row['Action_Plan']}")

        if found_actions:
            report += "\n".join(found_actions)
        else:
            report += "- General Security Alert: Avoid clicking links or sharing credentials."
    else:
        report += "STATUS: This message appears to be safe. No immediate action required."

    return report

###Verification: Testing with Real-World Scenarios
To ensure the integrity of our system, we perform an automated test using scenarios that include keywords defined in our remedies.csv. This validates that the engine correctly identifies the threat and retrieves the corresponding "First Aid" steps from the database. This section serves as our Prediction Analysis for project evaluation.

In [9]:
print("Available Keywords in CSV:", remedies_df['Keyword'].tolist())
print("-" * 50)

test_scenarios = [
    "Your bank card has been flagged for suspicious activity. Please verify your identity.",
    "Your social media password was changed. If this wasn't you, click here.",
    "URGENT: Download this security file to protect your computer immediately!",
]

for i, scenario in enumerate(test_scenarios):
    print(f"\nTEST SCENARIO {i+1}: {scenario}")
    result = phishaid_engine(scenario)
    print(result)
    print("-" * 50)

Available Keywords in CSV: ['card', 'transfer', 'iban', 'identity', 'password', 'instagram', 'hacked', 'wallet', 'airdrop', 'delivery', 'parcel', 'document', 'file', 'immediately']
--------------------------------------------------

TEST SCENARIO 1: Your bank card has been flagged for suspicious activity. Please verify your identity.
PHISHAID SECURITY DIAGNOSIS 
Classification: BENIGN (Confidence: 1.00)
Message Tone: Urgent / Negative (Panic-based)

STATUS: This message appears to be safe. No immediate action required.
--------------------------------------------------

TEST SCENARIO 2: Your social media password was changed. If this wasn't you, click here.
PHISHAID SECURITY DIAGNOSIS 
Classification: PHISHING (Confidence: 1.00)
Message Tone: Neutral / Informative

ALERT: THIS MESSAGE IS HIGHLY SUSPICIOUS!
EMERGENCY ACTION PLAN (from GitHub Database):
- [SocialMedia]: Reset your credentials, enable 2FA, and log out from all devices.
--------------------------------------------------

T

Gradio User Interface


In [17]:
import gradio as gr

#this function follows the same logic as phishaid_engine, but we also keep track of message history
def phishaid_logic(message, history):
    phish_res = phish_detector(message)[0]
    tone_res = sentiment_analyser(message)[0]

    prediction_label = phish_res['label'].lower()
    confidence_score = phish_res['score']

    tone_mapping = {
        "LABEL_0": "Urgent / Negative (Panic-based)",
        "LABEL_1": "Neutral / Informative",
        "LABEL_2": "Positive / Friendly"
    }

    friendly_tone = tone_mapping.get(tone_res['label'], "Undefined")

    is_phishing = prediction_label in ["phishing", "label_1"]
    bg_color = "#ffe6e6" if is_phishing else "#e6ffed"
    border_color = "#ff4d4d" if is_phishing else "#28a745"
    text_color = "#d9534f" if is_phishing else "#28a745"

    is_phishing = prediction_label in ["phishing", "label_1"]

    header = "### 🐠 PhishAid Security Diagnosis"
    status = f"**Classification:** {prediction_label.upper()} ({confidence_score:.2f} confidence)"
    tone = f"**Message Tone:** {friendly_tone}"

    report = f"{header}\n{status}\n{tone}\n\n---\n"

    if is_phishing:
        report += "###ALERT: THIS MESSAGE IS HIGHLY SUSPICIOUS!\n"
        report += "**EMERGENCY ACTION PLAN:**\n"

        found_actions = []
        # Căutăm în baza de date CSV
        for index, row in remedies_df.iterrows():
            if str(row['Keyword']).lower() in message.lower():
                found_actions.append(f"* **[{row['Category']}]:** {row['Action_Plan']}")

        if found_actions:
            report += "\n".join(found_actions)
        else:
            report += "* **General Security Alert:** Avoid clicking links or sharing credentials."
    else:
        report += "**STATUS:** This message appears to be safe. No immediate action required."

    return report


custom_css = ".gradio-container { background-color: #f7f9fc;} #title { text-align: center; color: #2c3e50; }"
with gr.Blocks(theme=gr.themes.Ocean()) as demo:
    gr.HTML("<h1 id='title'>🐠 PhishAid Assistant 𓆝 ⋆｡𖦹°‧</h1>")

    with gr.Sidebar():
        gr.Markdown("### 📊 Project Status")
        gr.Info("Phishing & Sentiment Models Loaded Successfully.")
        gr.Markdown("---")
        gr.Markdown("Using `remedies.csv` knowledge base.")

    gr.ChatInterface(
        fn=phishaid_logic,
        examples=["Your card has been blocked! Contact the bank immediately.", "Hello, check out this invoice."],
    )

if __name__ == "__main__":
    demo.launch()


/tmp/ipykernel_529/1985753711.py:53: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Ocean()) as demo:


Phishing & Sentiment Models Loaded Successfully.


/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://280e7c78a8be243871.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
